# 1 DATA PREPARATION

**Project:** Irish Milk Supply: Growth, Seasonality and Forecasting

**Author:** Temuulen Bulgan

### Purpose of this notebook
This notebook loads the raw data, checks its quality and turns it into one clean monthly table that the rest of the project uses for analysis, SQL and the Power BI dashboard.

### Data sources
| Dataset | Source | Content |
|---|---|---|
| AKM01 | CSO | Milk intake by creameries and pasteurisers; fat and protein content |
| AKM02 | CSO | Milk sales for human consumption (drinking milk) |
| AKM03 | CSO | Production of butter, cheese and skimmed milk powder |
| 32-10-0113-01 | Statistics Canada | Canadian milk production, used for comparison |

### Contents
1. AKM01: inspection and cleaning
2. AKM02: inspection and cleaning
3. AKM03: inspection and cleaning
4. Statistics Canada: inspection and cleaning
5. Combining into one monthly table
6. Saving the clean data

### Output
This notebook saves the cleaned data to `data/processed/ireland_dairy_monthly.csv` (one row per month, one column per measure).

## Setup

In [1]:
import pandas as pd
import numpy as np

# Show all columns when displaying a DataFrame
pd.set_option("display.max_columns", None)

## 1. AKM01: Milk intake, fat and protein content

**Source:** Central Statistics Office (CSO), table AKM01 "Intake of Cows Milk by Creameries and Pasteurisers" (now titled "Intake of Cows Milk by Milk Processors and Co-Ops"). Downloaded from data.cso.ie in December 2022. Licence: CC BY 4.0.

**Structure according to the source:** monthly figures for three statistics (milk intake, fat content and protein content), each split by Domestic and Import source.

In this section I inspect the raw file before cleaning it.

### 1.1 Inspection

#### Load the file and check its size

In [24]:
# Load the raw file
akm01 = pd.read_csv("../data/raw/AKM01.csv")

# Number of rows and columns
akm01.shape

(3444, 5)

#### First and last rows

In [25]:
# First and last 3 rows: format of the data, and where it starts and ends
display(akm01.head(3))
display(akm01.tail(3))

,Statistic Label,Month,Domestic or Import Source,UNIT,VALUE
0,Intake of Cows Milk by Creameries and Pasteuri...,1975 January,Domestic,Million Litres,55.8
1,Intake of Cows Milk by Creameries and Pasteuri...,1975 January,Import,Million Litres,NaN
2,Intake of Cows Milk by Creameries and Pasteuri...,1975 February,Domestic,Million Litres,84.4


,Statistic Label,Month,Domestic or Import Source,UNIT,VALUE
3441,Protein Content,2022 September,Import,%,NaN
3442,Protein Content,2022 October,Domestic,%,3.97
3443,Protein Content,2022 October,Import,%,NaN


First look: the data is in *long* format, with one value per row. The first rows are milk intake for January and February 1975, with one Domestic and one Import row per month. The last rows are protein content for 2022.

#### Column types and missing counts

In [26]:
# Column names, data types and non-missing counts
akm01.info()

<class 'pandas.DataFrame'>
RangeIndex: 3444 entries, 0 to 3443
Data columns (total 5 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   Statistic Label            3444 non-null   str  
 1   Month                      3444 non-null   str  
 2   Domestic or Import Source  3444 non-null   str  
 3   UNIT                       3444 non-null   str  
 4   VALUE                      1856 non-null   str  
dtypes: str(5)
memory usage: 134.7 KB


All five columns are stored as text (`str`). The first four columns are complete, but `VALUE` has only 1,856 non-missing values out of 3,444, and it is stored as text even though it should contain numbers.

#### Categories in each column

In [27]:
# Number of distinct values in each column
akm01.nunique()

Statistic Label                3
Month                        574
Domestic or Import Source      2
UNIT                           2
VALUE                        912
dtype: int64

In [28]:
# Categories in each descriptive column
print(akm01["Statistic Label"].value_counts(), "\n")
print(akm01["Domestic or Import Source"].value_counts(), "\n")
print(akm01["UNIT"].value_counts())

Statistic Label
Intake of Cows Milk by Creameries and Pasteurisers    1148
Fat Content                                           1148
Protein Content                                       1148
Name: count, dtype: int64 

Domestic or Import Source
Domestic    1722
Import      1722
Name: count, dtype: int64 

UNIT
%                 2296
Million Litres    1148
Name: count, dtype: int64


There are 3 statistics (1,148 rows each) and 2 sources (1,722 rows each). Milk intake is measured in million litres; fat and protein in %. `Month` has 574 distinct values (January 1975 to October 2022).

#### Check that every month is complete

In [29]:
# Number of rows for each month (first 3 shown)
akm01["Month"].value_counts().head(3)

Month
1975 January     6
1975 February    6
1975 March       6
Name: count, dtype: int64

In [30]:
# Check that every month has the same number of rows
akm01["Month"].value_counts().unique()

array([6])

In [31]:
# Check that no statistic, month and source combination appears twice
akm01.duplicated(subset=["Statistic Label", "Month", "Domestic or Import Source"]).sum()

np.int64(0)

Structure: all 574 months appear exactly 6 times (3 statistics × 2 sources). The file is a complete grid with no missing or duplicated rows: 574 × 6 = 3,444 rows.

#### Check the `VALUE` column

`VALUE` should contain numbers, but `.info()` shows it is stored as text. I check it in small steps.

In [32]:
# Count empty cells in VALUE in the original file
akm01["VALUE"].isna().sum()

np.int64(1588)

In [33]:
# Convert VALUE to numbers in a separate variable (akm01 is not changed)
# Anything that can't be converted becomes NaN
vals = pd.to_numeric(akm01["VALUE"], errors="coerce")

# Count missing values after the conversion
vals.isna().sum()

np.int64(1591)

After the conversion there are 3 more missing values than before (1,591 vs 1,588). So 3 cells contained text that could not be converted to a number.

In [34]:
# Show the values that were filled in but are not numbers
akm01.loc[vals.isna() & akm01["VALUE"].notna(), "VALUE"].value_counts()

VALUE
.    3
Name: count, dtype: int64

In [35]:
# Count usable numbers
vals.notna().sum()

np.int64(1853)

Summary of `VALUE`:
- 1,853 cells contain numbers
- 1,588 cells are empty
- 3 cells contain "." (CSO's symbol for "not available")
- Total: 3,444

Only the 3 dots stop `VALUE` from being numeric. They can be treated as missing values. All other filled cells are valid numbers.

#### Where the missing values are

I count the missing values for each statistic and source.

In [36]:
# Count missing values for each statistic and source
akm01[vals.isna()].groupby(["Statistic Label", "Domestic or Import Source"]).size()

Statistic Label                                     Domestic or Import Source
Fat Content                                         Domestic                      60
                                                    Import                       574
Intake of Cows Milk by Creameries and Pasteurisers  Import                       323
Protein Content                                     Domestic                      60
                                                    Import                       574
dtype: int64

Imported milk has no fat or protein data at all (574 missing each, which is every month). Imported milk intake has 323 missing months. Domestic fat and protein have 60 missing months each. Domestic milk intake does not appear in the list, so it has no missing values.

In [37]:
# Convert the Month text to real dates
dates = pd.to_datetime(akm01["Month"], format="%Y %B")
dates.head(3)

0   1975-01-01
1   1975-01-01
2   1975-02-01
Name: Month, dtype: datetime64[us]

In [38]:
# Keep only rows that have a number, and add the dates as a temporary column
with_data = akm01[vals.notna()].assign(date=dates)

# First and last month with data for each statistic and source
with_data.groupby(["Statistic Label", "Domestic or Import Source"])["date"].agg(["min", "max"])

min  \
Statistic Label                                    Domestic or Import Source              
Fat Content                                        Domestic                  1980-01-01   
Intake of Cows Milk by Creameries and Pasteurisers Domestic                  1975-01-01   
                                                   Import                    1998-01-01   
Protein Content                                    Domestic                  1980-01-01   

                                                                                    max  
Statistic Label                                    Domestic or Import Source             
Fat Content                                        Domestic                  2022-10-01  
Intake of Cows Milk by Creameries and Pasteurisers Domestic                  2022-10-01  
                                                   Import                    2019-09-01  
Protein Content                                    Domestic                  2022-10-01

Domestic milk intake covers the full period, January 1975 to October 2022. Domestic fat and protein start in January 1980, which explains their 60 missing months (1975–1979). Imported milk intake is only reported from January 1998 to September 2019. The 323 missing months are mostly outside that period (313 months), plus 10 gaps inside it. Imported fat and protein do not appear because they have no data at all.

#### Summary of findings

AKM01 has a complete structure: 574 months from 1975 to 2022, 6 rows per month, no duplicates. The `VALUE` column is text because of 3 "." entries. The missing values are not errors. They come from data CSO doesn't publish, mainly for imported milk. Domestic milk intake has no gaps.

For cleaning, I will convert `VALUE` and `Month` to the right types, keep only domestic data, and reshape the table to one row per month.

### 1.2 Cleaning

#### Convert `VALUE` to numbers

In [39]:
# Work on a copy so the raw data stays unchanged
akm01_clean = akm01.copy()

# Convert VALUE to numbers. The "." entries become NaN
akm01_clean["VALUE"] = pd.to_numeric(akm01_clean["VALUE"], errors="coerce")

akm01_clean["VALUE"].dtype

dtype('float64')

#### Convert `Month` to dates

In [40]:
# Convert the Month text to real dates in a new column
akm01_clean["date"] = pd.to_datetime(akm01_clean["Month"], format="%Y %B")

# Compare the original text and the new date
akm01_clean[["Month", "date"]].head(3)

,Month,date
0,1975 January,1975-01-01
1,1975 January,1975-01-01
2,1975 February,1975-02-01


#### Keep only domestic data
The import figures have no fat or protein data, and intake is only reported for 1998–2019, so I keep domestic data only.

In [41]:
# Keep only domestic rows
akm01_clean = akm01_clean[akm01_clean["Domestic or Import Source"] == "Domestic"]

# Check the result
print(akm01_clean.shape)
print(akm01_clean["Domestic or Import Source"].unique())

(1722, 6)
<StringArray>
['Domestic']
Length: 1, dtype: str


#### Reshape to one row per month
At the moment each month has 3 rows (intake, fat and protein). I turn the 3 statistics into columns, so each month becomes a single row.

In [42]:
# Turn the statistics into columns: one row per month
akm01_wide = akm01_clean.pivot(index="date", columns="Statistic Label", values="VALUE")

print(akm01_wide.shape)
akm01_wide.head(3)

(574, 3)


Statistic Label,Fat Content,Intake of Cows Milk by Creameries and Pasteurisers,Protein Content
date,,,
1975-01-01,NaN,55.8,NaN
1975-02-01,NaN,84.4,NaN
1975-03-01,NaN,193.3,NaN


#### Rename the columns
The CSO names are long, so I rename them to short names that include the unit.

In [43]:
# Short column names with units
akm01_wide = akm01_wide.rename(columns={
    "Intake of Cows Milk by Creameries and Pasteurisers": "intake_ml",
    "Fat Content": "fat_pct",
    "Protein Content": "protein_pct",
})

# Remove the leftover "Statistic Label" name above the columns
akm01_wide.columns.name = None

akm01_wide.head(3)

,fat_pct,intake_ml,protein_pct
date,,,
1975-01-01,NaN,55.8,NaN
1975-02-01,NaN,84.4,NaN
1975-03-01,NaN,193.3,NaN


#### Check the cleaned table

In [44]:
# Put the columns in a logical order
akm01_wide = akm01_wide[["intake_ml", "fat_pct", "protein_pct"]]

# Missing values in each column
akm01_wide.isna().sum()

intake_ml       0
fat_pct        60
protein_pct    60
dtype: int64

In [45]:
# Summary statistics to check the values make sense
akm01_wide.describe()

,intake_ml,fat_pct,protein_pct
count,574.000000,514.000000,514.000000
mean,450.669373,3.828677,3.331732
std,243.622290,0.329923,0.228793
min,55.800000,3.280000,2.860000
25%,217.125000,3.560000,3.180000
50%,456.600000,3.790000,3.280000
75%,637.700000,4.030000,3.470000
max,1181.000000,4.900000,4.040000


All values are in a realistic range. Monthly intake varies from 56 to 1,181 million litres, which reflects the strong seasonal pattern of Irish milk production. Fat is between 3.3% and 4.9% and protein between 2.9% and 4.0%.

#### Cleaning summary

I converted `VALUE` to numbers and `Month` to dates, kept only domestic data, and reshaped the table to one row per month. The result, `akm01_wide`, has 574 rows and 3 columns: `intake_ml`, `fat_pct` and `protein_pct`. Fat and protein are missing for 1975–1979 because CSO starts reporting them in 1980.

## 2. AKM02: Milk sales for human consumption

Source: Central Statistics Office (CSO), table AKM02 "Milk Sales (Dairy) for Human Consumption". Downloaded from data.cso.ie in December 2022. Licence: CC BY 4.0.

In this section I inspect the raw file before cleaning it.

### 2.1 Inspection

#### Load the file and check its size

In [46]:
# Load the raw file
akm02 = pd.read_csv("../data/raw/AKM02.csv")

# Number of rows and columns
akm02.shape

(2570, 5)

#### First and last rows

In [48]:
# First and last 3 rows: format of the data, and where it starts and ends
display(akm02.head(3))
display(akm02.tail(3))

,STATISTIC Label,Month,Type of Milk,UNIT,VALUE
0,Milk Sales (Dairy) for Human Consumption,1980 January,All Milk,Million Litres,38.2
1,Milk Sales (Dairy) for Human Consumption,1980 January,Whole Milk,Million Litres,38.2
2,Milk Sales (Dairy) for Human Consumption,1980 January,Skimmed and Semi-skimmed milk,Million Litres,NaN


,STATISTIC Label,Month,Type of Milk,UNIT,VALUE
2567,Milk Sales (Dairy) for Human Consumption,2022 October,Skimmed and Semi-skimmed milk,Million Litres,15.39
2568,Milk Sales (Dairy) for Human Consumption,2022 October,Skimmed Milk,Million Litres,NaN
2569,Milk Sales (Dairy) for Human Consumption,2022 October,Semi-skimmed milk,Million Litres,NaN


First look: the data is in long format. The first column is called `STATISTIC Label`. Instead of a source column there is `Type of Milk`. The data runs from January 1980 to October 2022, and the rows for each month are grouped together.

#### Column types and missing counts

In [49]:
# Column names, data types and non-missing counts
akm02.info()

<class 'pandas.DataFrame'>
RangeIndex: 2570 entries, 0 to 2569
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   STATISTIC Label  2570 non-null   str  
 1   Month            2570 non-null   str  
 2   Type of Milk     2570 non-null   str  
 3   UNIT             2570 non-null   str  
 4   VALUE            2262 non-null   str  
dtypes: str(5)
memory usage: 100.5 KB


All five columns are stored as text. The first four are complete, but `VALUE` has 2,262 non-missing values out of 2,570 and is stored as text, so it needs the same checks as in AKM01.

#### Categories in each column

In [50]:
# Number of distinct values in each column
akm02.nunique()

STATISTIC Label      1
Month              514
Type of Milk         5
UNIT                 1
VALUE              413
dtype: int64

In [51]:
# Categories in each descriptive column
print(akm02["STATISTIC Label"].value_counts(), "\n")
print(akm02["Type of Milk"].value_counts(), "\n")
print(akm02["UNIT"].value_counts())

STATISTIC Label
Milk Sales (Dairy) for Human Consumption    2570
Name: count, dtype: int64 

Type of Milk
All Milk                         514
Whole Milk                       514
Skimmed and Semi-skimmed milk    514
Skimmed Milk                     514
Semi-skimmed milk                514
Name: count, dtype: int64 

UNIT
Million Litres    2570
Name: count, dtype: int64


There is only one statistic (milk sales for human consumption) and one unit (million litres). `Type of Milk` has 5 categories with 514 rows each. "All Milk" is the total for all types. `Month` has 514 distinct values (January 1980 to October 2022).

#### Check that every month is complete

In [52]:
# Number of rows for each month (first 3 shown)
akm02["Month"].value_counts().head(3)

Month
1980 January     5
1980 February    5
1980 March       5
Name: count, dtype: int64

In [53]:
# Check that every month has the same number of rows
akm02["Month"].value_counts().unique()

array([5])

In [54]:
# Check that no month and milk type combination appears twice
akm02.duplicated(subset=["Month", "Type of Milk"]).sum()

np.int64(0)

Structure: all 514 months appear exactly 5 times (one row per milk type), with no duplicates. The file is a complete grid: 514 × 5 = 2,570 rows.

#### Check the `VALUE` column

`VALUE` should contain numbers, but `.info()` shows it is stored as text. I check it in the same steps as AKM01.


In [56]:
# Count empty cells in VALUE in the original file
akm02["VALUE"].isna().sum()

np.int64(308)

In [57]:
# Convert VALUE to numbers in a separate variable (akm02 is not changed)
# Anything that can't be converted becomes NaN
vals2 = pd.to_numeric(akm02["VALUE"], errors="coerce")

# Count missing values after the conversion
vals2.isna().sum()

np.int64(314)

In [58]:
# Show the values that were filled in but are not numbers
akm02.loc[vals2.isna() & akm02["VALUE"].notna(), "VALUE"].value_counts()

VALUE
.    6
Name: count, dtype: int64

In [59]:
# Count usable numbers
vals2.notna().sum()

np.int64(2256)

Summary of `VALUE`:
- 2,256 cells contain numbers
- 308 cells are empty
- 6 cells contain "."
- Total: 2,570

As in AKM01, only the "." entries stop `VALUE` from being numeric.

#### Where the missing values are

In [60]:
# Count missing values for each milk type
akm02[vals2.isna()].groupby("Type of Milk").size()

Type of Milk
Semi-skimmed milk                 85
Skimmed Milk                      85
Skimmed and Semi-skimmed milk    144
dtype: int64

In [61]:
# Convert the Month text to real dates
dates2 = pd.to_datetime(akm02["Month"], format="%Y %B")

# First and last month with data for each milk type
with_data2 = akm02[vals2.notna()].assign(date=dates2)
with_data2.groupby("Type of Milk")["date"].agg(["min", "max"])

,min,max
Type of Milk,,
All Milk,1980-01-01,2022-10-01
Semi-skimmed milk,1985-01-01,2020-09-01
Skimmed Milk,1985-01-01,2020-09-01
Skimmed and Semi-skimmed milk,1992-01-01,2022-10-01
Whole Milk,1980-01-01,2022-10-01


All Milk and Whole Milk are complete for 1980–2022. The other types were only reported for part of the period: Skimmed and Semi-skimmed separately from 1985 to September 2020, and combined from 1992. The missing values are the months outside these periods, so again they come from what CSO publishes, not from errors.

#### Summary of findings

AKM02 has a complete structure: 514 months from 1980 to 2022, 5 rows per month (one per milk type), no duplicates. The `VALUE` column is text because of 6 "." entries. All Milk is complete for the whole period. The other milk types are only reported for part of the period.

For cleaning, I will convert `VALUE` and `Month` to the right types and keep only All Milk, which is the total and has no gaps.

### 2.2 Cleaning

#### Keep only All Milk

In [62]:
# Work on a copy and keep only the All Milk rows
akm02_clean = akm02[akm02["Type of Milk"] == "All Milk"].copy()

akm02_clean.shape

(514, 5)

#### Convert `VALUE` to numbers and `Month` to dates

In [63]:
# Convert VALUE to numbers and Month to dates
akm02_clean["VALUE"] = pd.to_numeric(akm02_clean["VALUE"], errors="coerce")
akm02_clean["date"] = pd.to_datetime(akm02_clean["Month"], format="%Y %B")

akm02_clean[["Month", "date", "VALUE"]].head(3)

,Month,date,VALUE
0,1980 January,1980-01-01,38.2
5,1980 February,1980-02-01,36.6
10,1980 March,1980-03-01,38.4


#### Keep only the columns needed

In [65]:
# One row per month, with the date as the row label and a clear column name
akm02_wide = akm02_clean.set_index("date")[["VALUE"]].rename(columns={"VALUE": "liquid_sales_ml"})

akm02_wide.head(3)

,liquid_sales_ml
date,
1980-01-01,38.2
1980-02-01,36.6
1980-03-01,38.4


#### Check the cleaned table

In [66]:
# Missing values and summary statistics
print(akm02_wide.isna().sum(), "\n")
akm02_wide.describe()

liquid_sales_ml    0
dtype: int64 



,liquid_sales_ml
count,514.000000
mean,42.792412
std,2.301780
min,36.300000
25%,41.200000
50%,43.100000
75%,44.400000
max,48.200000


No missing values. Monthly sales of drinking milk are between 36 and 48 million litres, so they are quite stable compared with milk intake.

#### Cleaning summary

I kept only All Milk, converted `VALUE` and `Month` to the right types, and kept one column. The result, `akm02_wide`, has 514 rows (January 1980 to October 2022) and one column, `liquid_sales_ml`, with no missing values.